# Setup Environnement — StatParse (RAG_Statap_ENSAE_2025_Headminds)

Ce notebook configure un nouveau service Onyxia depuis zéro.
**Exécuter les cellules dans l'ordre.**

Durée estimée : **5-10 minutes** (hors téléchargement des modèles PaddleOCR ~3-5 min)

---

## Étape 1 — Dépendances système

In [ ]:
# poppler-utils : nécessaire pour pdf2image (pdfinfo, pdftoppm)
import subprocess
result = subprocess.run(["sudo", "apt", "install", "-y", "poppler-utils"],
                        capture_output=True, text=True)
print(result.stdout[-500:] if result.stdout else "")
print(result.stderr[-200:] if result.stderr else "")

# Vérification
r = subprocess.run(["which", "pdfinfo"], capture_output=True, text=True)
print(f"pdfinfo : {r.stdout.strip() or 'NON TROUVÉ'}")

## Étape 2 — Variables d'environnement PaddleOCR

Ces variables doivent être définies **avant** tout import de paddleocr/paddlex.

In [ ]:
import os
from pathlib import Path

# Chemin racine du dépôt (notebook lancé depuis la racine du projet)
REPO_ROOT = Path.cwd()

os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
os.environ["PADDLE_PDX_MODEL_SOURCE"] = "BOS"
print(f"REPO_ROOT : {REPO_ROOT}")
print("Variables d'environnement définies :")
print(f"  PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK = {os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK']}")
print(f"  PADDLE_PDX_MODEL_SOURCE = {os.environ['PADDLE_PDX_MODEL_SOURCE']}")

## Étape 3 — Dépendances Python (requirements.txt)

Installation via `uv pip` dans le venv Onyxia `/home/onyxia/work/.venv`.

In [ ]:
import subprocess

# Installer toutes les dépendances depuis requirements.txt
print("Installation depuis requirements.txt...")
r = subprocess.run(
    ["uv", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")],
    capture_output=True, text=True
)
status = "OK" if r.returncode == 0 else "ERREUR"
print(f"  [{status}] requirements.txt")
if r.returncode != 0:
    print(r.stderr[-500:])

# Remplacer opencv par la version headless (pas de GUI sur Onyxia)
print("Remplacement opencv \u2192 opencv-python-headless...")
r = subprocess.run(
    ["uv", "pip", "install", "opencv-python-headless"],
    capture_output=True, text=True
)
status = "OK" if r.returncode == 0 else "ERREUR"
print(f"  [{status}] opencv-python-headless")
if r.returncode != 0:
    print(r.stderr[-200:])

## Étape 4 — PaddleOCR (ordre strict obligatoire)

> **Important** : l'ordre d'installation est critique.
> `paddleocr==3.0.3` tire automatiquement `paddlex==3.5.0` mais il faut
> le redescendre à `paddlex==3.0.3` ensuite.

In [ ]:
import subprocess

paddle_packages = [
    ("paddlepaddle", "3.0.0"),
    ("paddleocr",    "3.0.3"),
    ("paddlex",      "3.0.3"),   # downgrade depuis 3.5.0 tiré par paddleocr
]

for pkg, version in paddle_packages:
    spec = f"{pkg}=={version}"
    print(f"Installation de {spec}...")
    r = subprocess.run(
        ["uv", "pip", "install", spec],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        # Extraire la ligne installed/upgraded du output
        lines = [l for l in r.stdout.split('\n') if pkg in l.lower()]
        print(f"  OK — {lines[-1].strip() if lines else 'installé'}")
    else:
        print(f"  ERREUR : {r.stderr[-300:]}")

print("\nVérification des versions installées :")
for pkg, expected in paddle_packages:
    r = subprocess.run(["uv", "pip", "show", pkg], capture_output=True, text=True)
    for line in r.stdout.split('\n'):
        if line.startswith('Version'):
            version_installed = line.split(':')[1].strip()
            ok = "✓" if version_installed == expected else "✗ ATTENDU " + expected
            print(f"  {pkg:15s} {version_installed:10s} {ok}")

## Étape 5 — Téléchargement des modèles PaddleOCR

Les modèles sont téléchargés une seule fois dans `~/.paddlex/official_models/`.
Durée : **3-5 minutes** selon la connexion.

In [ ]:
import os
import numpy as np

# Ces variables doivent être présentes avant l'import paddleocr
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
os.environ["PADDLE_PDX_MODEL_SOURCE"] = "BOS"

dummy = np.zeros((100, 300, 3), dtype=np.uint8)

print("1/2 — Téléchargement PaddleOCR PP-OCRv5 (texte)...")
from paddleocr import PaddleOCR
ocr = PaddleOCR(
    lang="ch",
    device="cpu",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)
list(ocr.predict(input=dummy))
print("    OK — modèles OCR texte téléchargés")

print("\n2/2 — Téléchargement PPStructureV3 (tables)...")
from paddleocr import PPStructureV3
table = PPStructureV3(
    device="cpu",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    use_table_recognition=True,
    use_formula_recognition=False,
    use_seal_recognition=False,
    use_chart_recognition=False,
)
list(table.predict(input=dummy))
print("    OK — modèles PPStructureV3 téléchargés")

print("\nTous les modèles sont dans ~/.paddlex/official_models/")

## Étape 6 — Vérification complète de l'environnement

In [ ]:
import subprocess, sys

checks = []

# pdfinfo
r = subprocess.run(["which", "pdfinfo"], capture_output=True, text=True)
checks.append(("poppler-utils (pdfinfo)", bool(r.stdout.strip()), r.stdout.strip()))

# numpy version
try:
    import numpy as np
    checks.append(("numpy", np.__version__ == "1.24.4", np.__version__))
except ImportError:
    checks.append(("numpy", False, "non installé"))

# opencv
try:
    import cv2
    checks.append(("opencv", True, cv2.__version__))
except ImportError:
    checks.append(("opencv", False, "non installé"))

# scikit-learn
try:
    import sklearn
    checks.append(("scikit-learn", True, sklearn.__version__))
except ImportError:
    checks.append(("scikit-learn", False, "non installé"))

# pymupdf
try:
    import fitz
    checks.append(("pymupdf (fitz)", True, fitz.__version__))
except ImportError:
    checks.append(("pymupdf (fitz)", False, "non installé"))

# paddlepaddle
try:
    import paddle
    checks.append(("paddlepaddle", paddle.__version__ == "3.0.0", paddle.__version__))
except ImportError:
    checks.append(("paddlepaddle", False, "non installé"))

# paddleocr
try:
    from paddleocr import PaddleOCR
    checks.append(("paddleocr", True, "3.0.3"))
except ImportError:
    checks.append(("paddleocr", False, "non installé"))

# paddlex
try:
    import paddlex
    checks.append(("paddlex", paddlex.__version__ == "3.0.3", paddlex.__version__))
except ImportError:
    checks.append(("paddlex", False, "non installé"))

# Modules statparse
import sys
sys.path.insert(0, str(REPO_ROOT))
for module in ["preprocessing", "segmentation", "classification",
               "reading_order", "ocr", "serialization"]:
    try:
        __import__(f"statparse.{module}")
        checks.append((f"statparse.{module}", True, "OK"))
    except Exception as e:
        checks.append((f"statparse.{module}", False, str(e)[:60]))

# Affichage
print(f"{'Composant':<35} {'Statut':^8} {'Version/Info'}")
print("-" * 75)
all_ok = True
for name, ok, info in checks:
    symbol = "✓" if ok else "✗"
    print(f"{name:<35} {symbol:^8} {info}")
    if not ok:
        all_ok = False

print()
if all_ok:
    print("Environnement complet — prêt à lancer le pipeline.")
else:
    print("Des composants manquent — relancer les étapes en erreur.")

## Étape 7 — Test rapide du pipeline end-to-end

Test sur un seul PDF pour confirmer que tout fonctionne de bout en bout.

In [ ]:
import sys, os, time
import numpy as np

os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"
os.environ["PADDLE_PDX_MODEL_SOURCE"] = "BOS"

sys.path.insert(0, str(REPO_ROOT))

from pathlib import Path
import fitz

from statparse.preprocessing import preprocess
from statparse.segmentation   import segment
from statparse.classification  import classify
from statparse.reading_order   import order_blocks
from statparse.ocr             import recognize_text
from statparse.serialization   import to_markdown

# Prendre le premier PDF disponible
pdf_dir = REPO_ROOT / "pdfs"
pdfs = list(pdf_dir.glob("*.pdf"))
if not pdfs:
    print("Aucun PDF trouvé dans pdfs/ — vérifier le dossier")
else:
    pdf_path = pdfs[0]
    print(f"PDF de test : {pdf_path.name}")
    print()

    t_total = time.perf_counter()

    # Rastérisation
    t0 = time.perf_counter()
    doc = fitz.open(str(pdf_path))
    pix = doc[0].get_pixmap(matrix=fitz.Matrix(150/72, 150/72))
    page_rgb = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)
    if pix.n == 4:
        import cv2; page_rgb = cv2.cvtColor(page_rgb, cv2.COLOR_RGBA2RGB)
    doc.close()
    print(f"  rasterize    : {time.perf_counter()-t0:.2f}s  ({page_rgb.shape[1]}x{page_rgb.shape[0]}px)")

    # Preprocessing
    t0 = time.perf_counter()
    binary = preprocess(page_rgb)
    print(f"  preprocess   : {time.perf_counter()-t0:.2f}s")

    # Segmentation
    t0 = time.perf_counter()
    blocks = segment(binary)
    print(f"  segment      : {time.perf_counter()-t0:.2f}s  ({len(blocks)} blocs)")

    # Classification
    t0 = time.perf_counter()
    labeled = classify(blocks, image_shape=binary.shape)
    from collections import Counter
    label_counts = Counter(b['label'] for b in labeled)
    print(f"  classify     : {time.perf_counter()-t0:.4f}s  {dict(label_counts)}")

    # Ordre de lecture
    t0 = time.perf_counter()
    ordered = order_blocks(labeled, image_shape=binary.shape)
    print(f"  reading_order: {time.perf_counter()-t0:.4f}s")

    # OCR
    t0 = time.perf_counter()
    text_blocks = recognize_text(ordered, page_rgb)
    n_chars = sum(len(b.get('text','')) for b in text_blocks)
    print(f"  ocr          : {time.perf_counter()-t0:.2f}s  ({n_chars} chars extraits)")

    # Sérialisation
    t0 = time.perf_counter()
    md = to_markdown(text_blocks)
    print(f"  serialize    : {time.perf_counter()-t0:.4f}s")

    print(f"\n  TOTAL        : {time.perf_counter()-t_total:.2f}s")
    print(f"\nMarkdown produit ({len(md)} chars) :")
    print("-" * 60)
    print(md[:500] + ("..." if len(md) > 500 else ""))

## Récapitulatif des commandes

Si tu préfères tout faire depuis le terminal en une seule fois :

```bash
# 1. Système
sudo apt install -y poppler-utils

# 2. Variables d'environnement (à répéter à chaque nouveau terminal)
export PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK=True
export PADDLE_PDX_MODEL_SOURCE=BOS

# 3. Dépendances Python
cd ~/work/RAG_Statap_ENSAE_2025_Headminds
uv pip install -r requirements.txt
uv pip install pymupdf opencv-python-headless

# 4. PaddleOCR — ordre strict
uv pip install paddlepaddle==3.0.0
uv pip install paddleocr==3.0.3
uv pip install paddlex==3.0.3

# 5. Télécharger les modèles
python scripts/download_models.py
```

### Notes importantes
- Les variables `PADDLE_PDX_*` doivent être définies **avant** tout import paddle
- `paddlex==3.0.3` doit être installé **après** `paddleocr==3.0.3` (qui tire 3.5.0)
- Les modèles sont dans `~/.paddlex/official_models/` et persistent tant que le service tourne
- À chaque **nouveau service** Onyxia : tout reprend de zéro (packages + modèles)